# Live Momentum Signal: BUY / SELL / HOLD

This notebook walks through `live_signal.py` step by step -- same signal logic and same
risk-management internals as the backtest (`momentum_backtest.py`), applied to live data
instead of historical data.

**Safety model:** everything below runs in **dry-run** mode -- orders are computed and logged
to `logs/live_trades_log.csv` but never sent to a broker. Real execution requires explicitly running
`live_signal.py --live` from the command line (see the last section), and paper-trading first is
strongly recommended before ever pointing this at a live account.

In [ ]:
# Epic 17: package is pip-installed editable, no sys.path hacking needed
from momentum_trading.execution.live_signal import (
    fetch_live_prices, calculate_period_returns, assign_ranks, get_top_etfs,
    compute_target_weights, generate_orders, log_orders, run,
)
from momentum_trading.backtest.momentum_backtest import BacktestConfig
from momentum_trading.core.paths import logs_dir
import pandas as pd

## 1. Configuration

Same `BacktestConfig` you validated in Notebook 3 -- reused unchanged, so live risk
management (drift threshold, position caps, regime filter, vol targeting) matches exactly
what was backtested.

In [5]:
# EDIT: your real universe (should match what Notebook 2's signal was trained/tested on)
tickers = ["SPY", "QQQ", "XLK", "XLF", "XLE", "XLY", "XLP", "XLU", "GLD", "TLT", "BIL"]

# EDIT: pull this from your broker (IBKR reqPositions) in production -- never trust local memory
current_holdings = {}   # e.g. {"SPY": 2, "XLK": 3}

# EDIT: pull real account net liquidation value from your broker in production
total_value = 1000.00

cfg = BacktestConfig(
    holding_period=1,
    initial_capital=total_value,
    commission=0.0,
    drift_threshold=0.03,
    min_trade_size=25.0,
    use_regime_filter=True,
    regime_benchmark="SPY",
    regime_sma_window=150,
    max_position_weight=0.35,
    allow_fractional_shares=True,
)

top_n = 10
lookback_period = 12

## 2. Fetch live prices

Pulls ~400 days of daily history through today via the same `functions.get_bulk_prices` path
used everywhere else in this project -- same vendor fallback chain, same formatting.

In [6]:
daily_prices = fetch_live_prices(tickers)
print(f"Fetched {daily_prices.shape[0]} days x {daily_prices.shape[1]} tickers")
daily_prices.tail()

2026-07-11 03:23:24,589 | INFO | Fetching live prices for 11 tickers, 2025-06-06 to 2026-07-11


Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK
Data retrieved from yf for XLF
Data retrieved from yf for XLE
Data retrieved from yf for XLY
Data retrieved from yf for XLP
Data retrieved from yf for XLU
Data retrieved from yf for GLD
Data retrieved from yf for TLT
Data retrieved from yf for BIL
Fetched 274 days x 11 tickers


,SPY,QQQ,XLK,XLF,XLE,XLY,XLP,XLU,GLD,TLT,BIL
date,,,,,,,,,,,
2026-07-06,751.280029,722.820007,183.570007,56.139999,53.130001,118.010002,84.099998,45.299999,382.130005,85.449997,91.430000
2026-07-07,747.710022,709.429993,179.179993,56.049999,54.639999,117.389999,84.860001,45.700001,377.489990,84.550003,91.449997
2026-07-08,745.400024,711.440002,181.399994,54.970001,55.599998,115.300003,84.389999,45.360001,374.450012,84.360001,91.449997
2026-07-09,751.710022,723.280029,185.350006,55.540001,54.820000,116.849998,83.199997,45.130001,378.179993,84.489998,91.459999
2026-07-10,754.950012,725.510010,185.779999,55.709999,55.080002,117.239998,84.120003,45.410000,377.010010,84.470001,91.500000


## 3. Compute today's signal

Identical `calculate_period_returns` / `assign_ranks` / `get_top_etfs` logic as Notebook 2,
just evaluated on live data through today instead of a fixed historical window.

In [7]:
monthly_prices = daily_prices.resample("ME").last()
scores = calculate_period_returns(monthly_prices, period=lookback_period).dropna(how="all")
ranks = assign_ranks(scores)
picks = get_top_etfs(ranks, top_n=top_n)
print("Today's picks:", picks)

Today's picks: ['XLK', 'XLE', 'QQQ', 'GLD', 'SPY', 'XLU', 'XLP', 'XLF', 'XLY', 'BIL']


## 4. Risk-managed target weights

Calls the SAME internal sizing functions (`_inverse_vol_weights`, `_apply_position_caps`) and
regime-filter logic from `momentum_backtest.py` -- not a reimplementation.

In [8]:
weights, gross_exposure = compute_target_weights(picks, daily_prices, cfg)
print(f"Gross exposure: {gross_exposure:.1%}")
pd.Series(weights, name="target_weight").sort_values(ascending=False)

2026-07-11 03:23:33,522 | INFO | Regime filter: SPY is above its 150D SMA -> scalar=1.00


Gross exposure: 100.0%


BIL    0.350000
SPY    0.103876
XLF    0.099615
XLP    0.087112
XLU    0.076770
XLY    0.070181
QQQ    0.057433
GLD    0.057174
XLE    0.055748
XLK    0.042091
Name: target_weight, dtype: float64

## 5. Generate BUY / SELL / HOLD orders

Compares target weights against `current_holdings`, applies `drift_threshold` and
`min_trade_size` -- same turnover-control logic validated in the backtest.

In [9]:
latest_prices = daily_prices.iloc[-1].to_dict()
orders = generate_orders(current_holdings, weights, gross_exposure, total_value, latest_prices, cfg)

orders_df = pd.DataFrame(orders).T
orders_df.index.name = "ticker"
orders_df

,action,shares,reason,rank,signal_score
ticker,,,,,
XLE,BUY,1.0121,drift $55.75,None,None
XLU,BUY,1.6905,drift $76.77,None,None
XLK,BUY,0.2265,drift $42.09,None,None
XLY,BUY,0.5986,drift $70.18,None,None
SPY,BUY,0.1375,drift $103.88,None,None
XLF,BUY,1.788,drift $99.61,None,None
BIL,BUY,3.8251,drift $350.00,None,None
XLP,BUY,1.0355,drift $87.11,None,None
GLD,BUY,0.1516,drift $57.17,None,None


## 6. Log the decision (audit trail, always happens -- even in dry-run)

In [ ]:
log_orders(orders, latest_prices, dry_run=True)
pd.read_csv(str(logs_dir() / "live_trades_log.csv")).tail(len(orders))

## 7. All-in-one: `run()` does steps 2-6 together

This is what the CLI script calls. Useful to schedule as a single function call.

In [11]:
orders = run(
    tickers=tickers,
    current_holdings=current_holdings,
    total_value=total_value,
    cfg=cfg,
    top_n=top_n,
    lookback_period=lookback_period,
    dry_run=True,   # NEVER set False from a notebook -- use the CLI with --live for real execution
)
orders

2026-07-11 03:23:45,699 | INFO | Fetching live prices for 11 tickers, 2025-06-06 to 2026-07-11


Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK
Data retrieved from yf for XLF
Data retrieved from yf for XLE


2026-07-11 03:23:46,061 | INFO | Today's signal picks (top 10): ['XLK', 'XLE', 'QQQ', 'GLD', 'SPY', 'XLU', 'XLP', 'XLF', 'XLY', 'BIL']
2026-07-11 03:23:46,068 | INFO | Regime filter: SPY is above its 150D SMA -> scalar=1.00
2026-07-11 03:23:46,069 | INFO | Target weights: {'XLK': 0.0420906773414927, 'XLE': 0.05574806895186332, 'QQQ': 0.05743318851627742, 'GLD': 0.05717441807561946, 'SPY': 0.10387617764840221, 'XLU': 0.07676991405265209, 'XLP': 0.0871117054014945, 'XLF': 0.09961486863164948, 'XLY': 0.07018098138054879, 'BIL': 0.35} | Gross exposure: 100.0%
2026-07-11 03:23:46,070 | INFO | XLE    BUY  shares=1    (drift $55.75)
2026-07-11 03:23:46,070 | INFO | XLU    BUY  shares=1    (drift $76.77)
2026-07-11 03:23:46,071 | INFO | XLK    BUY  shares=0    (drift $42.09)
2026-07-11 03:23:46,072 | INFO | XLY    BUY  shares=0    (drift $70.18)
2026-07-11 03:23:46,073 | INFO | SPY    BUY  shares=0    (drift $103.88)
2026-07-11 03:23:46,075 | INFO | XLF    BUY  shares=1    (drift $99.61)
2026-

Data retrieved from yf for XLY
Data retrieved from yf for XLP
Data retrieved from yf for XLU
Data retrieved from yf for GLD
Data retrieved from yf for TLT
Data retrieved from yf for BIL


{'XLE': {'action': 'BUY',
  'shares': np.float64(1.0121),
  'reason': 'drift $55.75',
  'rank': 2,
  'signal_score': 0.3025173530455536},
 'XLU': {'action': 'BUY',
  'shares': np.float64(1.6905),
  'reason': 'drift $76.77',
  'rank': 6,
  'signal_score': 0.08967734948985728},
 'XLK': {'action': 'BUY',
  'shares': np.float64(0.2265),
  'reason': 'drift $42.09',
  'rank': 1,
  'signal_score': 0.4215954034535747},
 'XLY': {'action': 'BUY',
  'shares': np.float64(0.5986),
  'reason': 'drift $70.18',
  'rank': 9,
  'signal_score': 0.06716252225645203},
 'SPY': {'action': 'BUY',
  'shares': np.float64(0.1375),
  'reason': 'drift $103.88',
  'rank': 5,
  'signal_score': 0.20762412803347297},
 'XLF': {'action': 'BUY',
  'shares': np.float64(1.788),
  'reason': 'drift $99.61',
  'rank': 8,
  'signal_score': 0.08028545373633889},
 'BIL': {'action': 'BUY',
  'shares': np.float64(3.8251),
  'reason': 'drift $350.00',
  'rank': 10,
  'signal_score': 0.03601939958654543},
 'XLP': {'action': 'BUY',
 

## 8. Going live (command line, NOT this notebook)

Real order placement is intentionally CLI-only, not notebook-callable, so a stray "Run All"
can never accidentally fire real trades.

```bash
# Dry run (safe, default) -- prints/logs orders, no broker call
python live_signal.py --dry-run

# Paper trading -- requires TWS/IB Gateway running, paper account, port 7497
python live_signal.py --live --port 7497

# Real money -- requires explicit double confirmation, verify your port number first
python live_signal.py --live --port 7496 --confirm-live-trading
```

Before ever running the last command: paper trade for at least 2-3 real rebalance cycles,
verify every fill matches the intended order, and confirm `current_holdings` is being pulled
from IBKR's real position feed (`reqPositions`), not from notebook memory.

---
## 9. Scheduling: is the market rebalance day today?

`is_rebalance_day()` uses the real NYSE calendar so you can schedule this script to run
**every day** via cron/Task Scheduler and let it self-gate, instead of hand-tracking holidays
and weekends yourself. The CLI already wires this in automatically (`--force` to override).

In [12]:
from momentum_trading.execution.live_signal import is_rebalance_day

print("Today is a scheduled rebalance day:", is_rebalance_day(holding_period_months=cfg.holding_period))
print("\nLinux cron (daily, self-gates):")
print('  35 9 * * 1-5 cd /path/to/nbs && /path/to/.venv/bin/python live_signal.py --dry-run >> logs/$(date +%Y%m%d).log 2>&1')
print("\nWindows Task Scheduler (CLI):")
print('  schtasks /Create /TN "MomentumSignal" /TR "C:\\path\\.venv\\Scripts\\python.exe C:\\path\\nbs\\live_signal.py --dry-run" /SC DAILY /ST 09:35 /RU SYSTEM')


Today is a scheduled rebalance day: False

Linux cron (daily, self-gates):
  35 9 * * 1-5 cd /path/to/nbs && /path/to/.venv/bin/python live_signal.py --dry-run >> logs/$(date +%Y%m%d).log 2>&1

Windows Task Scheduler (CLI):
  schtasks /Create /TN "MomentumSignal" /TR "C:\path\.venv\Scripts\python.exe C:\path\nbs\live_signal.py --dry-run" /SC DAILY /ST 09:35 /RU SYSTEM


---
## 10. Measuring REAL profit from the trade log

`measure_live_performance()` reads the actual `live_trades_log.csv` written by `log_orders()`
and computes FIFO-matched realized P&L plus mark-to-market unrealized P&L on open positions --
this is what genuinely happened (or would have, in dry-run), not a backtest re-simulation.

In [ ]:
from momentum_trading.execution.live_signal import measure_live_performance

# EDIT: your real start/end dates and current market prices for open positions
start_date = "2026-01-01"
end_date = "2026-12-31"

perf = measure_live_performance(
    start_date=start_date,
    end_date=end_date,
    latest_prices=latest_prices,   # from step 5 above -- swap for a fresh fetch if run later
    log_path=str(logs_dir() / "live_trades_log.csv"),
    initial_capital=total_value,
)

print(f"Realized P&L:   ${perf['realized_pnl']:,.2f}")
print(f"Unrealized P&L: ${perf['unrealized_pnl']:,.2f}")
print(f"Total P&L:      ${perf['total_pnl']:,.2f}")
if 'total_return_pct' in perf:
    print(f"Total Return:   {perf['total_return_pct']:.2%}")
print(f"\nOpen positions: {perf['open_positions']}")
print(f"Trade count: {perf['trade_count']}")
pd.Series(perf['per_ticker_realized'], name='realized_pnl').sort_values(ascending=False)

---
## 11. Multiple portfolios, same strategy

`run_multi_portfolio()` runs the identical strategy/config independently across separate ticker
universes -- each portfolio gets its own signal, sizing, orders, and its own trade log file, so
P&L never mixes between portfolios.

In [15]:
from momentum_trading.execution.live_signal import run_multi_portfolio

# EDIT: your real portfolio definitions
portfolios = {
    "portfolio1": ["SPY", "QQQ", "XLK"],
    "portfolio2": ["XLF", "XLE", "GLD", "TLT"],
}

# Single float applies the same value to every portfolio; or pass a dict per-portfolio:
# total_value_per_portfolio = {"portfolio1": 1000.0, "portfolio2": 2500.0}
multi_results = run_multi_portfolio(
    portfolios=portfolios,
    total_value_per_portfolio=1000.0,
    cfg=cfg,
    top_n=2,
    dry_run=True,   # NEVER set False from a notebook -- use the CLI with --live
)

for name, orders in multi_results.items():
    print(f"\n--- {name} ---")
    display(pd.DataFrame(orders).T)

print("\nEach portfolio's decisions are logged separately: live_trades_log_<portfolio_name>.csv")
print("Measure each one independently with measure_live_performance(..., log_path='live_trades_log_portfolio1.csv')")


2026-07-11 03:24:30,676 | INFO | ========================================
2026-07-11 03:24:30,678 | INFO | Portfolio: portfolio1 | tickers=['SPY', 'QQQ', 'XLK'] | value=$1000.00 | custom_weights=no (algorithmic sizing)
2026-07-11 03:24:30,679 | INFO | Fetching live prices for 3 tickers, 2025-06-06 to 2026-07-11
2026-07-11 03:24:30,811 | INFO | Today's signal picks (top 2): ['XLK', 'QQQ']
2026-07-11 03:24:30,815 | INFO | Regime filter: SPY is above its 150D SMA -> scalar=1.00
2026-07-11 03:24:30,816 | INFO | Target weights: {'XLK': 0.5, 'QQQ': 0.5} | Gross exposure: 100.0%
2026-07-11 03:24:30,818 | INFO | XLK    BUY  shares=2    (drift $500.00)
2026-07-11 03:24:30,820 | INFO | QQQ    BUY  shares=0    (drift $500.00)
2026-07-11 03:24:30,830 | INFO | Logged 2 order decisions to live_trades_log_portfolio1.csv (config_hash=9329a2dcbb53)
2026-07-11 03:24:30,831 | INFO | DRY RUN: no orders sent to broker. Use --live to actually trade.
2026-07-11 03:24:30,833 | INFO | =========================

Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK
Determining best data source using XLF...
Data retrieved from yf for XLF
Using yf for all tickers


2026-07-11 03:24:30,991 | INFO | Today's signal picks (top 2): ['XLE', 'GLD']
2026-07-11 03:24:30,995 | INFO | Target weights: {'XLE': 0.5, 'GLD': 0.5} | Gross exposure: 100.0%
2026-07-11 03:24:30,995 | INFO | XLE    BUY  shares=9    (drift $500.00)
2026-07-11 03:24:30,996 | INFO | GLD    BUY  shares=1    (drift $500.00)
2026-07-11 03:24:31,007 | INFO | Logged 2 order decisions to live_trades_log_portfolio2.csv (config_hash=9329a2dcbb53)
2026-07-11 03:24:31,008 | INFO | DRY RUN: no orders sent to broker. Use --live to actually trade.


Data retrieved from yf for XLF
Data retrieved from yf for XLE
Data retrieved from yf for GLD
Data retrieved from yf for TLT

--- portfolio1 ---


,action,shares,reason,rank,signal_score
XLK,BUY,2.6913,drift $500.00,1,0.421595
QQQ,BUY,0.6891,drift $500.00,2,0.29025



--- portfolio2 ---


,action,shares,reason,rank,signal_score
XLE,BUY,9.0777,drift $500.00,1,0.302517
GLD,BUY,1.3262,drift $500.00,2,0.244422



Each portfolio's decisions are logged separately: live_trades_log_<portfolio_name>.csv
Measure each one independently with measure_live_performance(..., log_path='live_trades_log_portfolio1.csv')


---
## 12. Custom weights (bypass algorithmic sizing)

Both `run()` and `run_multi_portfolio()` accept an optional `custom_weights` dict --
when provided, inverse-vol sizing and the correlation penalty are skipped entirely and
your specified weights are used directly (still subject to `max_position_weight` capping).

In [16]:
# Single-portfolio custom weights
my_weights = {"SPY": 0.5, "QQQ": 0.3, "XLK": 0.2}

orders_custom = run(
    tickers=list(my_weights.keys()),
    current_holdings=current_holdings,
    total_value=total_value,
    cfg=cfg,
    dry_run=True,
    custom_weights=my_weights,
)
orders_custom

2026-07-11 03:24:37,771 | INFO | Fetching live prices for 3 tickers, 2025-06-06 to 2026-07-11
2026-07-11 03:24:37,948 | INFO | Today's signal picks (top 10): ['XLK', 'QQQ', 'SPY']
2026-07-11 03:24:37,950 | INFO | Regime filter: SPY is above its 150D SMA -> scalar=1.00


Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK


2026-07-11 03:24:37,950 | INFO | Target weights: {'SPY': 0.35, 'QQQ': 0.35, 'XLK': 0.30000000000000004} | Gross exposure: 100.0%
2026-07-11 03:24:37,951 | INFO | XLK    BUY  shares=1    (drift $300.00)
2026-07-11 03:24:37,953 | INFO | QQQ    BUY  shares=0    (drift $350.00)
2026-07-11 03:24:37,954 | INFO | SPY    BUY  shares=0    (drift $350.00)
2026-07-11 03:24:37,957 | INFO | Logged 3 order decisions to live_trades_log.csv (config_hash=9329a2dcbb53)
2026-07-11 03:24:37,958 | INFO | DRY RUN: no orders sent to broker. Use --live to actually trade.


{'XLK': {'action': 'BUY',
  'shares': np.float64(1.6148),
  'reason': 'drift $300.00',
  'rank': 1,
  'signal_score': 0.4215954034535747},
 'QQQ': {'action': 'BUY',
  'shares': np.float64(0.4824),
  'reason': 'drift $350.00',
  'rank': 2,
  'signal_score': 0.29025001644459403},
 'SPY': {'action': 'BUY',
  'shares': np.float64(0.4636),
  'reason': 'drift $350.00',
  'rank': 3,
  'signal_score': 0.20762412803347297}}

In [17]:
# Multi-portfolio: mix algorithmic and custom-weighted portfolios in one call
portfolios_mixed = {
    "algo_portfolio": {"tickers": ["SPY", "QQQ", "XLK"], "custom_weights": None},
    "custom_portfolio": {"tickers": ["XLF", "XLE", "GLD", "TLT"],
                          "custom_weights": {"XLF": 0.4, "XLE": 0.3, "GLD": 0.2, "TLT": 0.1}},
}

mixed_results = run_multi_portfolio(
    portfolios=portfolios_mixed,
    total_value_per_portfolio=1000.0,
    cfg=cfg,
    top_n=3,
    dry_run=True,
)
for name, orders in mixed_results.items():
    print(f"\n--- {name} ---")
    display(pd.DataFrame(orders).T)

2026-07-11 03:25:45,263 | INFO | ========================================
2026-07-11 03:25:45,265 | INFO | Portfolio: algo_portfolio | tickers=['SPY', 'QQQ', 'XLK'] | value=$1000.00 | custom_weights=no (algorithmic sizing)
2026-07-11 03:25:45,266 | INFO | Fetching live prices for 3 tickers, 2025-06-06 to 2026-07-11
2026-07-11 03:25:45,385 | INFO | Today's signal picks (top 3): ['XLK', 'QQQ', 'SPY']
2026-07-11 03:25:45,392 | INFO | Regime filter: SPY is above its 150D SMA -> scalar=1.00
2026-07-11 03:25:45,393 | INFO | Target weights: {'XLK': 0.3, 'QQQ': 0.35, 'SPY': 0.35} | Gross exposure: 100.0%
2026-07-11 03:25:45,394 | INFO | XLK    BUY  shares=1    (drift $300.00)
2026-07-11 03:25:45,396 | INFO | SPY    BUY  shares=0    (drift $350.00)
2026-07-11 03:25:45,397 | INFO | QQQ    BUY  shares=0    (drift $350.00)
2026-07-11 03:25:45,448 | INFO | Logged 3 order decisions to live_trades_log_algo_portfolio.csv (config_hash=9329a2dcbb53)
2026-07-11 03:25:45,449 | INFO | DRY RUN: no orders se

Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK


2026-07-11 03:25:45,452 | INFO | Fetching live prices for 4 tickers, 2025-06-06 to 2026-07-11
2026-07-11 03:25:45,569 | INFO | Today's signal picks (top 3): ['XLE', 'GLD', 'XLF']
2026-07-11 03:25:45,570 | INFO | Target weights: {'XLF': 0.35, 'XLE': 0.35, 'GLD': 0.30000000000000004} | Gross exposure: 100.0%
2026-07-11 03:25:45,571 | INFO | XLE    BUY  shares=6    (drift $350.00)
2026-07-11 03:25:45,573 | INFO | GLD    BUY  shares=0    (drift $300.00)
2026-07-11 03:25:45,574 | INFO | XLF    BUY  shares=6    (drift $350.00)


Determining best data source using XLF...
Data retrieved from yf for XLF
Using yf for all tickers
Data retrieved from yf for XLF
Data retrieved from yf for XLE
Data retrieved from yf for GLD
Data retrieved from yf for TLT


2026-07-11 03:25:47,922 | INFO | Logged 3 order decisions to live_trades_log_custom_portfolio.csv (config_hash=9329a2dcbb53)
2026-07-11 03:25:47,924 | INFO | DRY RUN: no orders sent to broker. Use --live to actually trade.



--- algo_portfolio ---


,action,shares,reason,rank,signal_score
XLK,BUY,1.6148,drift $300.00,1,0.421595
SPY,BUY,0.4636,drift $350.00,3,0.207624
QQQ,BUY,0.4824,drift $350.00,2,0.29025



--- custom_portfolio ---


,action,shares,reason,rank,signal_score
XLE,BUY,6.3543,drift $350.00,1,0.302517
GLD,BUY,0.7957,drift $300.00,2,0.244422
XLF,BUY,6.2825,drift $350.00,3,0.080285
